In [49]:
! pip install pandas
! pip install openpyxl

In [50]:
import pandas as pd
import json

In [51]:
file_path = '../data/xl/dados_metas_atualizados_v2.xlsx'

In [52]:
df = pd.read_excel(file_path, engine='openpyxl', sheet_name='Temas, Planos e ODS')

In [53]:
df.head()

,Número da Meta,Destaque do Título,Descrição da Meta,Projeção,Indicador,Secretaria,Eixo,Temas,Subprefeituras,ODS,Nº da Meta ODS,Planos Municipais
0,1,Entregar 8 novos parques,"Entregar 8 novos parques, para ampliar a ofert...","Com essa medida, a cidade chegará a 125 parque...",Número de parques entregues.,SVMA,Universo SP,Natureza e ambiente,Campo Limpo; Casa Verde/Cachoeirinha; Cidade A...,6; 11; 13; 15,11.7; 15.1; 13.1,Plano Diretor Estratégico; Plano Municipal de ...
1,2,Requalificar ou ampliar 25 parques,"Requalificar ou ampliar 25 parques, com a melh...",Serão priorizadas as unidades com indicadores ...,Número de parques revitalizados.,SVMA,Universo SP,Natureza e ambiente,Butantã; Capão Redondo; Ermelino Matarazzo; It...,11; 12; 13; 15,11.7; 15.1; 13.1,Plano Diretor Estratégico; Plano Municipal de ...
2,3,Realizar o projeto São Paulo +Verde,"Realizar o projeto São Paulo +Verde, que vai p...",A iniciativa envolve o plantio de árvores em p...,Número de árvores plantadas.,SVMA; SMSUB,Universo SP,Natureza e ambiente,NaN,2; 6; 11; 12; 13; 15,2.5; 11.7; 13.1; 15.1; 15.2,Plano Municipal de Arborização Urbana
3,4,"Alcançar 11,4% de tratamento de materiais reci...","Alcançar 11,4% de tratamento de materiais reci...",O aumento da capacidade de separação de resídu...,Percentual de resíduos recicláveis não destina...,SP REGULA; SMSUB; SMDET,Universo SP,Natureza e ambiente,NaN,6; 7; 8; 12,12.5; 16.6,Plano de Gestão Integrada de Resíduos Sólidos
4,5,Levar atividades de conscientização e educação...,Levar atividades de conscientização e educação...,Além de cursos e atividades formativas da Univ...,Número de atendimentos em ações de educação am...,SVMA,Universo SP,Natureza e ambiente,NaN,4; 12,3.6; 9.1; 16.6,Plano Municipal de Educação Ambiental


In [54]:
df.columns

Index(['Número da Meta', 'Destaque do Título', 'Descrição da Meta', 'Projeção',
       'Indicador', 'Secretaria', 'Eixo', 'Temas', 'Subprefeituras', 'ODS',
       'Nº da Meta ODS', 'Planos Municipais'],
      dtype='object')

In [55]:
renomear = {
    'Número da Meta' : 'numero',
    'Destaque do Título' : 'destaque',
    'Descrição da Meta' : 'descricao',
    'Projeção' : 'projecao',
    'Indicador' : 'indicador',
    'Secretaria' : 'orgao_responsavel',
    'Eixo' : 'eixo'
}

In [56]:
df.rename(columns=renomear, inplace=True)

In [57]:
df_metas = df[renomear.values()].copy().reset_index(drop=True)

In [58]:
df_metas.head()

,numero,destaque,descricao,projecao,indicador,orgao_responsavel,eixo
0,1,Entregar 8 novos parques,"Entregar 8 novos parques, para ampliar a ofert...","Com essa medida, a cidade chegará a 125 parque...",Número de parques entregues.,SVMA,Universo SP
1,2,Requalificar ou ampliar 25 parques,"Requalificar ou ampliar 25 parques, com a melh...",Serão priorizadas as unidades com indicadores ...,Número de parques revitalizados.,SVMA,Universo SP
2,3,Realizar o projeto São Paulo +Verde,"Realizar o projeto São Paulo +Verde, que vai p...",A iniciativa envolve o plantio de árvores em p...,Número de árvores plantadas.,SVMA; SMSUB,Universo SP
3,4,"Alcançar 11,4% de tratamento de materiais reci...","Alcançar 11,4% de tratamento de materiais reci...",O aumento da capacidade de separação de resídu...,Percentual de resíduos recicláveis não destina...,SP REGULA; SMSUB; SMDET,Universo SP
4,5,Levar atividades de conscientização e educação...,Levar atividades de conscientização e educação...,Além de cursos e atividades formativas da Univ...,Número de atendimentos em ações de educação am...,SVMA,Universo SP


In [59]:
df_metas['orgao_responsavel'] = df_metas['orgao_responsavel'].str.replace('SP REGULA', 'SPRegula')
df_metas['orgao_responsavel'] = df_metas['orgao_responsavel'].str.replace('SP URBANISMO', 'SPUrbanismo')
df_metas['orgao_responsavel'] = df_metas['orgao_responsavel'].str.replace('FUNDACAO PAULISTANA', 'FPD')
df_metas['orgao_responsavel'] = df_metas['orgao_responsavel'].str.replace('SPTURIS', 'SPTuris')
df_metas['orgao_responsavel'] = df_metas['orgao_responsavel'].str.replace('SPTRANS', 'SPTrans')
df_metas['orgao_responsavel'] = df_metas['orgao_responsavel'].str.replace('SP OBRAS', 'SPObras')
df_metas['orgao_responsavel'] = df_metas['orgao_responsavel'].str.replace('SUBPREFEITURAS', 'SMSUB')

In [60]:
df_metas['numero'].duplicated().any()

np.False_

In [61]:
df_metas.shape

(132, 7)

In [62]:
metas_parsed = []

for i, row in df_metas.iterrows():


    orgaos = row['orgao_responsavel'].split(';')
    orgaos = [orgao.strip() for orgao in orgaos]

    eixo = row['eixo'].title()

    if eixo.endswith('Sp'):
        eixo = eixo[:-2] + 'SP'

    meta = {
        'numero': row['numero'],
        'destaque': row['destaque'],
        'descricao': row['descricao'],
        'projecao': row['projecao'],
        'indicador': row['indicador'],
        'orgao_responsavel' : orgaos,
        'eixo': eixo
    }

    metas_parsed.append(meta)

In [63]:
metas_parsed[:3]

[{'numero': 1,
  'destaque': 'Entregar 8 novos parques',
  'descricao': 'Entregar 8 novos parques, para ampliar a oferta de áreas de lazer e fortalecer a cultura de convivência harmoniosa entre as pessoas e a natureza.',
  'projecao': 'Com essa medida, a cidade chegará a 125 parques municipais, entre eles o Parque da Joia, no Butantã, um dos mais pedidos pela população ao longo do ciclo participativo do PdM. A área já abrange nascentes, lago, agrofloresta, praça e espaços de lazer que, com a conclusão das obras pela Prefeitura, deverão fortalecer na região a recuperação ecológica, a preservação da biodiversidade local, a integração comunitária e a educação ambiental.',
  'indicador': 'Número de parques entregues.',
  'orgao_responsavel': ['SVMA'],
  'eixo': 'Universo SP'},
 {'numero': 2,
  'destaque': 'Requalificar ou ampliar 25 parques',
  'descricao': 'Requalificar ou ampliar 25 parques, com a melhoria das instalações e para maior segurança dos frequentadores.',
  'projecao': 'Serão 

In [64]:
fpath = '../data/metas.json'

with open(fpath, 'w', encoding='utf-8') as f:
    json.dump(metas_parsed, f, ensure_ascii=False, indent=4)